In [ ]:
%pip install -r ../requirements.txt
# pip install --force-reinstall numpy==1.18.5

In [ ]:
import os, shutil, zipfile, requests

url = "https://github.com/francescopausellii/uniform-coloring-ai/releases/download/v1.0/emnist.zip"
src = "../emnist.zip"
dst = os.path.expanduser("~/.cache/emnist/emnist.zip")

os.makedirs(os.path.dirname(dst), exist_ok=True)

# download se non esiste o è corrotto
if not os.path.exists(src) or not zipfile.is_zipfile(src):
    print("Scarico EMNIST...")
    with open(src, "wb") as f:
        f.write(requests.get(url).content)

# copia in cache
shutil.copy(src, dst)


In [ ]:
import numpy as np
import pandas as pd
import keras
from keras import layers
import cv2 as cv
from matplotlib import pyplot as plt

# Percorso modelli (relativo a notebooks/)
MODELS_DIR = "../models/"

# Durata del training (in epoche)
TRAINING_EPOCHS = 25

# Risoluzione del parametro 'rho' (la distanza dall'origine alla linea) in pixel.
# Un valore pari a 1 indica la massima precisione possibile (1 pixel).
HOUGH_RHO = 1

# Risoluzione del parametro 'theta' (l'angolo della linea) in radianti.
# Dividendo Pi Greco per 180 otteniamo l'equivalente di 1 grado (1°).
HOUGH_THETA = np.pi / 180  # risoluzione angolare (1°)

# Numero minimo di intersezioni (voti) nello "spazio di Hough" affinché una linea
# venga considerata valida. Più è basso, più l'algoritmo sarà sensibile e troverà
# anche linee deboli.
HOUGH_THRESHOLD = 60

# Lunghezza minima che deve avere un segmento per essere rilevato.
HOUGH_MIN_LEN = 40

# Distanza massima consentita tra due segmenti posizionati sulla stessa linea per unirli
HOUGH_MAX_GAP = 10

# Tolleranza in gradi per classificazione direzione segmenti
ANGLE_TOL_DEG = 15

# Se due segmenti paralleli sono < di questa distanza, allora sono considerati sulla stessa linea
POSITION_TOL_PX = 20

In [ ]:
from emnist import extract_training_samples
from emnist import extract_test_samples

# x_train conterrà le matrici dei pixel delle immagini.
# y_train conterrà le etichette corrispondenti a ciascuna immagine.
x_train, y_train = extract_training_samples("balanced")

# Valutazione performance del modello
# su dati mai visti durante l'addestramento.
x_test, y_test = extract_test_samples("balanced")

# print forma dataset
print("Forma train_images: ", x_train.shape)
# forma immagine
print("Forma dell'immagine:", x_train[0].shape)

# print range valori pixel immagini
print("Max =", np.max(x_train), "  Min =", np.min(x_train))

# visualizzazione dataset

index = 0

plt.imshow(x_train[index], cmap=plt.cm.binary)
print(y_train[index])

In [ ]:
# tutte le labels o valori presenti nel dataset scelto
ALL_LABELS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 
          'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
          'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't']

# lettere scelte per filtrare il dataset 
SELECTED_LABELS = ['B', 'G', 'T', 'Y']  

def filterDataset(X_data, y_data):
    # Controllo tipo etichette
    assert y_data.dtype == np.uint8
    classes = ALL_LABELS
    new_data_size = 0
    for recordIndex in range(0, y_data.shape[0]):
        currentLabel = classes[y_data[recordIndex]]
        if currentLabel in SELECTED_LABELS:
            new_data_size += 1

    new_X_data = np.zeros((new_data_size, 28, 28), dtype = X_data.dtype)
    new_y_data = np.zeros((new_data_size,), dtype = np.uint8)

    new_data_index = 0
    for recordIndex in range(0, y_data.shape[0]):
        currentLabel = classes[y_data[recordIndex]]

        if currentLabel not in SELECTED_LABELS:
            continue
        
        new_X_data[new_data_index] = X_data[recordIndex]
        new_y_data[new_data_index] = SELECTED_LABELS.index(currentLabel)
        new_data_index += 1
    
    assert new_data_index == new_X_data.shape[0]
    return (new_X_data, new_y_data)

Filtraggio dataset per tenere solo le lettere che ci interessano

In [ ]:
(x_train_filter, y_train_filter) = filterDataset(x_train, y_train)
(x_test_filter, y_test_filter) = filterDataset(x_test, y_test)

# normalizzazione per calcolo dei pesi più veloce
x_train_filter = x_train_filter / np.max(x_train_filter)
x_test_filter = x_test_filter / np.max(x_test_filter)

# Stampa informazioni dataset filtrato
print("Train:", x_train_filter.shape, " Test:", x_test_filter.shape)

# Stampa distribuzione classi nel dataset filtrato
# Quante immagini per ciascuna classe sono presenti nel dataset di addestramento filtrato
print(
    "Distribuzione classi train:",
    {
        SELECTED_LABELS[k]: int(v)
        for k, v in zip(*np.unique(y_train_filter, return_counts=True))
    },
)

In [ ]:
# visualizzazione dataset filtrato
index = 0

plt.imshow(x_train_filter[index], cmap=plt.cm.binary)
print(SELECTED_LABELS[y_train_filter[index]])

# Creazione Modello

Modello: MLP (fully-connected)

Un MLP non ha l'invarianza spaziale di una CNN: ogni pixel è un input indipendente, quindi una lettera leggermente spostata/ruotata/scalata rispetto agli esempi EMNIST produce attivazioni diverse.
Per compensare si usa la **data augmentation**: rotazioni, traslazioni e zoom casuali applicati solo durante il training.

In [ ]:
from pathlib import Path

DENSE_PATH = MODELS_DIR + "dense.keras"

if Path(DENSE_PATH).exists():
    model_dense = keras.models.load_model(DENSE_PATH)
else:
    model_dense = keras.Sequential(
        [
            # Immagini 28x28 con 1 canale (scala di grigi)
            layers.Input(shape=(28, 28, 1)),
            # Variazioni casuali per rendere il modello più robusto a lettere disegnate a mano
            layers.RandomRotation(0.06),  # ±~22 gradi
            layers.RandomTranslation(0.12, 0.12),  # ±12% in x e y
            layers.RandomZoom(0.15),  # ±15%
            layers.Flatten(),
            # Usa relu perché è più veloce da calcolare
            layers.Dense(256, activation="relu"),
            # Evita overfitting durante il training, disattiva casualmente il 30% dei neuroni
            layers.Dropout(0.3),
            layers.Dense(64, activation="relu"),
            # Softmax per classificazione multiclasse, output un vettore di probabilità per ciascuna classe
            layers.Dense(len(SELECTED_LABELS), activation="softmax"),
        ]
    )

    # Configuriamo il modello
    # adam: l'ottimizzatore che regola i pesi in base agli errori.
    # sparse_categorical_crossentropy: la funzione di perdita ideale quando i target sono numeri interi.
    # accuracy: la metrica che useremo per monitorare la percentuale di risposte corrette.
    model_dense.compile(
        optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
    )

    # epoche extra rispetto al default: l'augmentation forte rallenta la convergenza
    # Tengo da parte il 10% dei dati per validazione
    model_dense.fit(
        x_train_filter[..., np.newaxis], y_train_filter, epochs=TRAINING_EPOCHS, validation_split=0.1
    )
    model_dense.save(DENSE_PATH)

model_dense.summary()

In [ ]:
# Valutazione del modello sul test set
x_test_in = x_test_filter[..., np.newaxis]
result_dense = model_dense.evaluate(x_test_in, y_test_filter)

In [ ]:
# modello usato dalla pipeline di riconoscimento celle (predict_cell)
model = model_dense

## Visualizzazione dati test

visualizzazione degli errori effettuati dal modello 

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_predicted = model.predict(x_test_filter[..., np.newaxis])
y_predicted_labels = np.argmax(y_predicted, axis=1)

# Confronta le etichette vere con quelle predette e costruisce una matrice di confusione
# normalize='true': ogni riga somma a 1 -> percentuali per classe vera
confusionMatrix = confusion_matrix(y_test_filter, y_predicted_labels, normalize="true")

# accuracy per classe: diagonale della matrice normalizzata
for lbl, acc in zip(SELECTED_LABELS, confusionMatrix.diagonal()):
    print(f"Accuracy {lbl}: {acc:.2%}")

# Grafico della matrice di confusione (righe = classe vera, colonne = classe predetta)
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusionMatrix, display_labels=SELECTED_LABELS
)
disp.plot()
plt.show()

# Individuazione Griglia e riconoscimento celle

In [ ]:
def preparation(path):

    img = cv.imread(path)
    if img is None:
        raise FileNotFoundError(f"Immagine non trovata: {path}")
    print(f"Immagine: {img.shape[1]}x{img.shape[0]} px")

    # Non servono i canali di colore, solo grigi
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    # Sfocature per migliore rilevamento bordi
    blurred = cv.GaussianBlur(gray, (5, 5), 0)

    # Canny con soglie automatiche (metodo Otsu)
    otsu_thresh, _ = cv.threshold(blurred, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

    # Rilevamento bordi con Canny usando le soglie calcolate da Otsu
    edges = cv.Canny(blurred, threshold1=otsu_thresh * 0.5, threshold2=otsu_thresh)

    _show([img, gray, edges], ["Originale", "CLAHE gray", "Canny edges"])

    return img, gray, edges


def extract_grid_mask(gray):
    """
    Isola SOLO le linee della griglia, eliminando i tratti delle lettere.
    Otsu globale (immagini sintetiche pulite, niente speckle da adaptive),
    poi erosione+dilatazione con kernel lungo 1D: sopravvive solo cio' che
    ha molti pixel continui in una direzione -> linee griglia, non lettere.
    """

    # Immagine convertita in bianco e nero
    _, bw = cv.threshold(gray, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

    h, w = gray.shape

    # Creazione rettangolo lungo 
    hk = cv.getStructuringElement(cv.MORPH_RECT, (w // 10, 1))
    # Erosione per rimuovere le lettere
    hor = cv.dilate(cv.erode(bw, hk), hk)

    # Creazione rettangolo alto
    vk = cv.getStructuringElement(cv.MORPH_RECT, (1, h // 10))
    ver = cv.dilate(cv.erode(bw, vk), vk)

    # Maschera finale come somma di linee orizzontali e verticali
    grid = cv.add(hor, ver)
    _show(
        [bw, hor, ver, grid], ["Binaria Otsu", "Linee H", "Linee V", "Maschera griglia"]
    )
    return grid


In [ ]:
def detect_segments(edges):

    # Cerca segmenti partendo dai bordi rilevati da Canny
    segments = cv.HoughLinesP(
        edges,
        rho=HOUGH_RHO,
        theta=HOUGH_THETA,
        threshold=HOUGH_THRESHOLD,
        minLineLength=HOUGH_MIN_LEN,
        maxLineGap=HOUGH_MAX_GAP,
    )
    if segments is None:
        raise ValueError("Nessun segmento trovato")

    # Converto in lista di tuple
    segments = [tuple(s[0]) for s in segments]
    print(f"Segmenti Hough trovati: {len(segments)}")
    return segments


# controllo angoli dei segmenti per classificazionne orizzontale o verticale
def segment_angle_deg(x1, y1, x2, y2):
    angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
    if angle < -90:
        angle += 180
    if angle >= 90:
        angle -= 180
    return angle


def classify_segments(segs, angle_tol=ANGLE_TOL_DEG):

    horiz, vert = [], []
    for seg in segs:
        a = segment_angle_deg(*seg)
        # Se l'angolo è vicino a 0° → orizzontale, se è vicino a ±90° → verticale
        if abs(a) <= angle_tol:
            horiz.append(seg)
        elif abs(abs(a) - 90) <= angle_tol:
            vert.append(seg)
        # segmenti obliqui (lettere diagonali) vengono ignorati

    print(f"Segmenti orizzontali: {len(horiz)}")
    print(f"Segmenti verticali:   {len(vert)}")
    return horiz, vert


def segment_representative_position(segs, axis):
    """
    Per ogni segmento calcola la posizione "trasversale" media, per vedere la posizione della linea nello spazio:
      - linee orizzontali → media di y1, y2  (axis='y')
      - linee verticali   → media di x1, x2  (axis='x')
    Restituisce lista di (posizione, segmento).
    """
    result = []
    for s in segs:
        x1, y1, x2, y2 = s
        pos = (y1 + y2) / 2 if axis == "y" else (x1 + x2) / 2
        result.append((pos, s))
    return result


def cluster_lines_by_position(pos_segs, tol=POSITION_TOL_PX):
    """
    Raggruppa segmenti con posizione trasversale simile (entro `tol` pixel).
    Ogni gruppo rappresenta UNA linea della griglia.
    Restituisce lista di cluster, ciascuno = lista di segmenti.
    """
    sorted_ps = sorted(pos_segs, key=lambda x: x[0])
    clusters = []
    current = [sorted_ps[0]]
    for ps in sorted_ps[1:]:
        # Se la posizione è vicina all'ultima del cluster corrente, aggiungi, altrimenti inizia nuovo cluster
        if ps[0] - current[-1][0] <= tol:
            current.append(ps)
        else:
            clusters.append(current)
            current = [ps]
    clusters.append(current)
    return clusters


In [ ]:
def fit_line_from_cluster(cluster, img_shape):
    """
    Dati tutti i punti dei segmenti di un cluster, fa una regressione lineare
    (np.polyfit) e restituisce due punti estremi (p1, p2) che attraversano
    l'intera immagine.
    Questo è il 'averaging' dei punti che elimina l'effetto delle linee storte.
    """

    # Prendo tutti i punti estremi dei segmenti del cluster
    pts = []
    for _, (x1, y1, x2, y2) in cluster:
        pts.append((x1, y1))
        pts.append((x2, y2))

    pts = np.array(pts, dtype=float)
    h, w = img_shape[:2]

    # Decide se fittare y=f(x) o x=f(y) in base alla direzione prevalente
    span_x = pts[:, 0].max() - pts[:, 0].min()
    span_y = pts[:, 1].max() - pts[:, 1].min()

    if span_x >= span_y:
        coeffs = np.polyfit(
            pts[:, 0], pts[:, 1], 1
        )  # Regressione lineare di grado 1 -> restituisce la pendenza (a) e l'intercetta (b)
        x_start, x_end = (
            0,
            w - 1,
        )  # Estende la linea da sinistra a destra per tutta l'immagine
        y_start = int(
            np.polyval(coeffs, x_start)
        )  # Calcola l'altezza Y all'inizio dello schermo
        y_end = int(
            np.polyval(coeffs, x_end)
        )  # Calcola l'altezza Y alla fine dello schermo
        return (x_start, y_start), (x_end, y_end), coeffs, "h"

    else:  # linea prevalentemente verticale
        coeffs = np.polyfit(pts[:, 1], pts[:, 0], 1)  # x = a*y + b
        y_start, y_end = (
            0,
            h - 1,
        )  # Estende la linea dall'alto in basso per tutta l'immagine
        x_start = int(np.polyval(coeffs, y_start))
        x_end = int(np.polyval(coeffs, y_end))
        return (x_start, y_start), (x_end, y_end), coeffs, "v"


def line_intersection(line_h, line_v):
    """
    Intersezione esatta tra una linea orizzontale e una verticale,
    entrambe espresse come (coeffs, tipo).
    line_h: coeffs per y = a*x + b
    line_v: coeffs per x = a*y + b
    Restituisce (x, y) float.
    """

    # Calcola le equazioni delle linee e risolve il sistema per trovare l'intersezione
    a_h, b_h = line_h  # y = a_h * x + b_h
    a_v, b_v = line_v  # x = a_v * y + b_v
    # Sostituendo: y = a_h*(a_v*y + b_v) + b_h
    # y*(1 - a_h*a_v) = a_h*b_v + b_h
    denom = 1 - a_h * a_v
    if abs(denom) < 1e-9:  # linee parallele (non dovrebbe succedere)
        return None

    # Coordinate del punto di intersezione
    y = (a_h * b_v + b_h) / denom
    x = a_v * y + b_v
    return (x, y)


def build_grid_lines(horiz_clusters, vert_clusters, img_shape):
    """
    Per ogni cluster calcola la linea media.
    Ordina le linee orizzontali per y crescente, verticali per x crescente.
    Restituisce h_lines, v_lines (liste di dizionari con p1, p2, coeffs, tipo)
    """
    h_lines, v_lines = [], []

    # Trasforma i segmenti orizzontali in rette continue e calcola la posizione media per ordinare
    for cl in horiz_clusters:
        p1, p2, coeffs, tipo = fit_line_from_cluster(cl, img_shape)
        mid_y = (p1[1] + p2[1]) / 2
        h_lines.append(
            {"p1": p1, "p2": p2, "coeffs": coeffs, "tipo": tipo, "pos": mid_y}
        )

    # Trasforma i segmenti verticali in rette continue e calcola la posizione media per ordinare
    for cl in vert_clusters:
        p1, p2, coeffs, tipo = fit_line_from_cluster(cl, img_shape)
        mid_x = (p1[0] + p2[0]) / 2
        v_lines.append(
            {"p1": p1, "p2": p2, "coeffs": coeffs, "tipo": tipo, "pos": mid_x}
        )

    # Ordinamento linee orizzontali per posizione y e verticali per posizione x
    h_lines.sort(key=lambda l: l["pos"])
    v_lines.sort(key=lambda l: l["pos"])

    print(f"Linee griglia orizzontali: {len(h_lines)}")
    print(f"Linee griglia verticali:   {len(v_lines)}")
    return h_lines, v_lines


def compute_intersections(h_lines, v_lines):
    """
    Calcola tutte le intersezioni tra linee H e V.
    Restituisce matrice numpy (n_h, n_v, 2) di coordinate (x, y).
    """
    n_h, n_v = len(h_lines), len(v_lines)

    # Matrice di punti di intersezione
    pts = np.zeros((n_h, n_v, 2), dtype=float)

    # Calcola le intersezioni tra tutte le linee orizzontali e verticali
    for i, hl in enumerate(h_lines):
        for j, vl in enumerate(v_lines):
            pt = line_intersection(hl["coeffs"], vl["coeffs"])
            if pt is None:
                pt = (0, 0)
            pts[i, j] = pt
    return pts


In [ ]:
def extract_cell_from_intersections(gray, pts, row, col, padding=16, target=(128, 128)):
    """
    Estrae la cella (row, col) usando i 4 angoli calcolati dalle intersezioni.
    Applica una prospettiva corretta (warpPerspective) per raddrizzare celle storte.

    La cella viene estratta ad ALTA risoluzione (128x128): la riduzione a 28x28
    avviene solo in prepare_cell_for_emnist, DOPO la binarizzazione. Warpando
    direttamente a 28x28 i tratti delle lettere si riducono a 1-2 px sfumati e
    la soglia adattiva + pulizia morfologica li distruggono (es. la B perdeva
    la spina verticale e veniva classificata come Y).
    """
    # Prende i 4 angoli della cella dalla matrice di intersezioni
    tl = pts[row, col]  # top-left
    tr = pts[row, col + 1]  # top-right
    bl = pts[row + 1, col]  # bottom-left
    br = pts[row + 1, col + 1]  # bottom-right

    # Forma reale della cella nella prospettiva originale
    src = np.array([tl, tr, br, bl], dtype=np.float32)

    # Forma rettangolare per la cella estratta
    dst = np.array(
        [
            [0, 0],
            [target[0] - 1, 0],
            [target[0] - 1, target[1] - 1],
            [0, target[1] - 1],
        ],
        dtype=np.float32,
    )

    # Matrice di trasformazione prospettica per mappare punti src in dst
    M = cv.getPerspectiveTransform(src, dst)
    cell = cv.warpPerspective(gray, M, target)

    # Padding interno: rimuove i residui delle linee di griglia sul bordo
    if padding > 0:
        cell = cell[padding:-padding, padding:-padding]
        cell = cv.resize(cell, target, interpolation=cv.INTER_AREA)

    return cell


def extract_all_cells(gray, pts):
    """
    Estrae tutte le celle dalla matrice di intersezioni.
    pts ha shape (n_h, n_v, 2) → griglia di (n_h-1) x (n_v-1) celle.
    Restituisce cells_2d: lista 2D di array numpy.
    """
    # Con n_h linee orizzontali e n_v linee verticali, ci sono (n_h-1) x (n_v-1) celle
    n_rows = pts.shape[0] - 1
    n_cols = pts.shape[1] - 1
    cells_2d = []

    for r in range(n_rows):
        row_cells = []
        for c in range(n_cols):
            # Singola cella estratta
            cell = extract_cell_from_intersections(gray, pts, r, c)
            row_cells.append(cell)
        cells_2d.append(row_cells)

    print(f"Celle estratte: {n_rows} x {n_cols}")
    return cells_2d


In [ ]:
def _show(imgs, titles=None, cmap="gray"):
    """
    Utility per visualizzare più immagini affiancate con titoli opzionali
    """
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        disp = cv.cvtColor(img, cv.COLOR_BGR2RGB) if len(img.shape) == 3 else img
        ax.imshow(disp, cmap=cmap if len(img.shape) == 2 else None)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i], fontsize=10)
    plt.tight_layout()
    plt.show()


def draw_grid_lines(img, h_lines, v_lines):
    """Visualizza le linee della griglia sull'immagine originale"""
    vis = img.copy()
    # Linee orizzontali in verde, verticali in blu
    for l in h_lines:
        cv.line(vis, l["p1"], l["p2"], (0, 220, 80), 2)
    for l in v_lines:
        cv.line(vis, l["p1"], l["p2"], (0, 100, 255), 2)
    _show([vis], ["Linee griglia rilevate (verde=H, blu=V)"])


def draw_intersections(img, pts):
    """Visualizza i punti di intersezione tra le linee della griglia"""
    vis = img.copy()
    # Punti di intersezione in rosso
    for i in range(pts.shape[0]):
        for j in range(pts.shape[1]):
            x, y = int(pts[i, j, 0]), int(pts[i, j, 1])
            cv.circle(vis, (x, y), 5, (0, 0, 255), -1)
    _show([vis], ["Intersezioni griglia"])


def draw_all_segments(img, segs, horiz, vert):
    """Visualizza tutti i segmenti rilevati, evidenziando quelli orizzontali e verticali"""
    vis = img.copy()
    # Segmenti in grigio chiaro, orizzontali in verde, verticali in blu
    for s in segs:
        cv.line(vis, s[:2], s[2:], (150, 150, 150), 1)
    for s in horiz:
        cv.line(vis, s[:2], s[2:], (0, 200, 80), 2)
    for s in vert:
        cv.line(vis, s[:2], s[2:], (0, 100, 255), 2)
    _show([vis], ["Segmenti Hough (grigio=scartati, verde=H, blu=V)"])


def visualize_cells(cells_2d):
    """Visualizza tutte le celle estratte in una griglia"""
    n_r = len(cells_2d)
    n_c = len(cells_2d[0]) if cells_2d else 0
    fig, axes = plt.subplots(n_r, n_c, figsize=(n_c * 1.3, n_r * 1.3))
    if n_r == 1 and n_c == 1:
        axes = np.array([[axes]])
    elif n_r == 1:
        axes = axes[np.newaxis, :]
    elif n_c == 1:
        axes = axes[:, np.newaxis]
    for r in range(n_r):
        for c in range(n_c):
            axes[r, c].imshow(cells_2d[r][c], cmap="gray")
            axes[r, c].axis("off")
            axes[r, c].set_title(f"{r},{c}", fontsize=6)
    plt.suptitle("Celle estratte", fontweight="bold")
    plt.tight_layout()
    plt.show()


def print_matrix(matrix):
    print("\nMATRICE RICONOSCIUTA:")
    cols = len(matrix[0]) if matrix else 0
    print("   " + "  ".join(str(j) for j in range(cols)))
    print("   " + "──" * cols)
    for i, row in enumerate(matrix):
        print(f"{i:2} │ " + "  ".join(str(c) for c in row))
    print()


In [ ]:
def prepare_cell_for_emnist(cell_gray, target_size=(28, 28)):
    """
    Trasforma una cella grezza (grayscale uint8, alta risoluzione) nel formato EMNIST.

    Pipeline:
      1. Denoising adattivo
      2. Binarizzazione → lettera bianca su sfondo nero (come EMNIST)
      3. Pulizia morfologica + filtro componenti connesse
      4. Bounding box del carattere → crop + padding
      5. Resize a target_size con aspect ratio preservato
      6. Normalizzazione [0.0, 1.0]

    Input:  numpy array (H, W) uint8
    Output: numpy array (28, 28) float32, pronto per model.predict()
    """
    steps = {}  # per debug

    # Rimuove le imperfezioni della scrittura
    # h=12: più alto per immagini più rumorose, più basso per immagini pulite. Se è troppo alto sfuma i tratti sottili.
    denoised = cv.fastNlMeansDenoising(
        cell_gray, h=12, templateWindowSize=7, searchWindowSize=21
    )
    steps["1_denoised"] = denoised

    # Converte l'immagine in bianco e nero
    # blockSize adattivo: deve restare piu' grande dello spessore del tratto
    # (~8-10 px su celle 128x128), altrimenti l'interno del tratto viene
    # scambiato per sfondo e la lettera si svuota.
    bs = max(17, (min(cell_gray.shape) // 5) | 1)  # dispari, >= 17
    binary = cv.adaptiveThreshold(
        denoised,
        255,
        cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY_INV,  # INV → lettera bianca, sfondo nero (EMNIST)
        blockSize=bs,
        C=8,
    )
    steps["2_binary"] = binary

    # Elimina pixel isolati e chiude piccoli buchi nel tratto.
    kernel = np.ones((2, 2), np.uint8)
    cleaned = cv.morphologyEx(
        binary, cv.MORPH_OPEN, kernel, iterations=1
    )  # toglie rumore
    cleaned = cv.morphologyEx(
        cleaned, cv.MORPH_CLOSE, kernel, iterations=1
    )  # chiude buchi
    steps["3_cleaned"] = cleaned

    # Nelle griglie disegnate a mano le linee sono ondulate e il padding del
    # crop non basta a eliminarle: frammenti di linea entrano nella cella,
    # allargano il bounding box e schiacciano la lettera in un angolo.
    # Teniamo la componente piu' grande (la lettera) e le componenti che NON
    # toccano il bordo e hanno area >= 5% della maggiore (tratti staccati
    # della stessa lettera); i frammenti di griglia toccano sempre il bordo.
    n_cc, cc_labels, stats, _ = cv.connectedComponentsWithStats(cleaned, connectivity=8)
    if n_cc > 2:  # piu' di una componente oltre lo sfondo
        H_cc, W_cc = cleaned.shape
        areas = stats[1:, cv.CC_STAT_AREA]
        main = 1 + int(np.argmax(areas))
        keep = np.zeros_like(cleaned)
        for i in range(1, n_cc):
            x0, y0, w0, h0, a0 = stats[i]
            touches_border = x0 == 0 or y0 == 0 or x0 + w0 >= W_cc or y0 + h0 >= H_cc
            if i == main or (not touches_border and a0 >= 0.05 * areas.max()):
                keep[cc_labels == i] = 255
        cleaned = keep
    steps["3b_components"] = cleaned

    # Trova il rettangolo minimo che contiene tutti i pixel bianchi.
    # Se la cella è vuota (nessun pixel bianco) restituisce un array di zeri.
    coords = cv.findNonZero(cleaned)
    if coords is None:
        # Cella vuota (lettera assente o completamente rumore)
        return np.zeros(target_size, dtype=np.float32)

    x, y, w, h = cv.boundingRect(coords)

    # Margine di sicurezza: 15% della dimensione maggiore, minimo 2px
    margin = max(2, int(max(w, h) * 0.15))
    H_cell, W_cell = cleaned.shape
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(W_cell, x + w + margin)
    y2 = min(H_cell, y + h + margin)

    cropped = cleaned[y1:y2, x1:x2]
    steps["4_cropped"] = cropped

    # EMNIST centra il carattere in una canvas 28×28 con bordo.
    # Facciamo lo stesso: resize a max 20×20, poi padding centrato a 28×28.
    inner = target_size[0] - 8  # 20px: lascia 4px di bordo per lato
    ch, cw = cropped.shape
    scale = inner / max(ch, cw)
    new_h = max(1, int(ch * scale))
    new_w = max(1, int(cw * scale))
    resized = cv.resize(cropped, (new_w, new_h), interpolation=cv.INTER_AREA)

    # Padding centrato
    canvas = np.zeros(target_size, dtype=np.uint8)
    pad_y = (target_size[1] - new_h) // 2
    pad_x = (target_size[0] - new_w) // 2
    canvas[pad_y : pad_y + new_h, pad_x : pad_x + new_w] = resized
    steps["5_canvas"] = canvas

    # Normalizza i pixel
    normalized = canvas.astype(np.float32) / 255.0

    return normalized

In [ ]:
def predict_cell(cell_img):
    # Prepara la cella per il modello EMNIST
    ready = prepare_cell_for_emnist(cell_img)  # (28, 28) float32

    # adatta input alla forma attesa dal modello caricato
    shape = model.input_shape[1:]  # es (28,28,1) o (784,)
    inp = ready.reshape((1,) + tuple(shape))

    # predice la lettera usando il modello addestrato, restituisce quella con probabilità più alta
    probs = model.predict(inp, verbose=0)[0]
    return SELECTED_LABELS[np.argmax(probs)]


In [ ]:
image_path = "../grid_imgs/image2.png"

# Carica l'immagine, la converte in scala di grigi e ne estrae i contorni (Canny)
print("STEP 1: Preprocessing")
img, gray, edges = preparation(image_path)

# Rileva segmenti di linee con Hough, classificandoli in orizzontali e verticali
print("\nSTEP 2: Rilevamento segmenti Hough")
grid = extract_grid_mask(gray)
segs = detect_segments(grid)

# Classifica i segmenti in orizzontali e verticali, scartando quelli obliqui
print("\nSTEP 3: Classificazione H/V")
horiz, vert = classify_segments(segs)

# Mostra tutti i segmenti rilevati, evidenziando quelli orizzontali e verticali
draw_all_segments(img, segs, horiz, vert)

# Calcola la coordinata trasversale (X o Y) di ogni singolo frammento
print("\nSTEP 4: Clustering e linee medie")
horiz_ps = segment_representative_position(horiz, "y")
vert_ps = segment_representative_position(vert, "x")
horiz_cl = cluster_lines_by_position(horiz_ps)
vert_cl = cluster_lines_by_position(vert_ps)


# Scarta cluster generati dai tratti delle lettere (es. gamba della R,
# spina della B): una vera linea di griglia attraversa quasi tutta la
# griglia, un tratto di lettera resta dentro UNA cella.
def _cluster_span(cluster, coord):
    vals = [v for _, s in cluster for v in (s[coord], s[coord + 2])]
    return max(vals) - min(vals)


# Soglia relativa all'estensione della griglia (non dell'immagine): la
# griglia puo' occupare solo una parte del frame, quindi confrontiamo lo
# span di ogni cluster con lo span massimo tra i cluster dello stesso asse.
SPAN_FRAC = 0.6
if vert_cl:
    max_v_span = max(_cluster_span(cl, 1) for cl in vert_cl)
    vert_cl = [cl for cl in vert_cl if _cluster_span(cl, 1) >= SPAN_FRAC * max_v_span]
if horiz_cl:
    max_h_span = max(_cluster_span(cl, 0) for cl in horiz_cl)
    horiz_cl = [cl for cl in horiz_cl if _cluster_span(cl, 0) >= SPAN_FRAC * max_h_span]
print(f"Linee valide dopo filtro span -> H:{len(horiz_cl)} V:{len(vert_cl)}")
h_lines, v_lines = build_grid_lines(horiz_cl, vert_cl, img.shape)

draw_grid_lines(img, h_lines, v_lines)

# Calcola le intersezioni tra le linee della griglia per ottenere i vertici delle celle
print("\nSTEP 5: Calcolo intersezioni")
pts = compute_intersections(h_lines, v_lines)

draw_intersections(img, pts)

# Calcola quante celle ci sono in totale (es. se ci sono 4 linee verticali, ci sono 3 colonne)
n_celle_r = pts.shape[0] - 1
n_celle_c = pts.shape[1] - 1
print(f"Griglia: {n_celle_r} righe × {n_celle_c} colonne")

# 6. Estrazione + classificazione
print("\nSTEP 6: Estrazione celle e classificazione")
# Ritaglia, raddrizza in prospettiva ed elimina i bordi neri da ogni quadratino
cells_2d = extract_all_cells(gray, pts)

visualize_cells(cells_2d)

matrix = []
for r, row_cells in enumerate(cells_2d):
    # Passa ogni singola immagine ritagliata alla funzione che si interfaccia con la rete neurale
    labels = [predict_cell(cell) for cell in row_cells]
    matrix.append(labels)
    print(f"  Riga {r}: {labels}")

print("=" * 55)
print_matrix(matrix)